# TrashScan — Path A simplificado

Este notebook assume que a estrutura **já está baixada/criada no volume** do RunPod:

- `/workspace/.venv`
- `/workspace/TrashScan`
- `/workspace/TACO`
- `/workspace/external_datasets`
- `/workspace/processed_4cls`
- `/workspace/runs`
- `/workspace/results_path_A`

O foco aqui é rodar o fluxo do **Path A** usando os scripts `.py` do projeto, sem refazer downloads e sem refazer merge/preprocessamento por padrão.

> Observação: os pesos dos modelos **`yolov11m`**, **`yolov8m`** e **`yolov9s`** já foram calculados e devem estar disponíveis em `runs/path_A`.


## 0) Antes de abrir/rodar o notebook

No terminal do pod, ative a venv do volume:

```bash
cd /workspace
source .venv/bin/activate
```

Se precisar reinstalar dependências:

```bash
cd /workspace/TrashScan/env
python -m pip install -r environment.txt
```

No VSCode/Jupyter, selecione o kernel correspondente a:

```bash
/workspace/.venv/bin/python
```

Para confirmar dentro do notebook, rode a célula abaixo e confira `sys.executable`.


## 1) Imports, paths e utilitários

In [ ]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'
DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'

# Treino fica no disco local do pod para evitar I/O lento e cache pesado no volume.
# Se o pod for deletado, salve o essencial de volta no volume antes.
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A_5cls)'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume
RESULTS_PATH_A_DIR = WORKSPACE / 'results_path_A_5cls'

for p in [RUNS_PATH_A_DIR, MLFLOW_DIR, RESULTS_PATH_A_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_A_DIR =', RESULTS_PATH_A_DIR)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)

    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


## 2) Verificação dos arquivos principais

In [ ]:
required_paths = [
    DATA_DIR / 'merge_datasets.py',
    DATA_DIR / 'preprocess.py',
    TRAIN_DIR / 'train_path_A.py',
    EVAL_DIR / 'evaluate.py',
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print('Arquivos ausentes:')
    for m in missing:
        print(' -', m)
    raise FileNotFoundError('Há scripts ausentes no clone. Veja a lista acima.')

print('Todos os scripts principais foram encontrados.')
print('Dataset YAML existe?', DATASET_YAML_PATH_A.exists())


## 3) Configuração de GPU e treino

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    DEVICE = '0'
else:
    gpu_name = 'cpu'
    DEVICE = 'cpu'

# Ajuste se necessário
EPOCHS = 100
BATCH = 16
IMGSZ = 640
PATIENCE = 30

# Pesos já calculados anteriormente: yolov11m, yolov8m, yolov9s.
# Se rodar a célula de treino abaixo, esses modelos serão treinados novamente.
MODELS = [
    'yolov11m',
    'yolov8m',
    'yolov9s',
]

print('Dispositivo:', gpu_name)
print('DEVICE     :', DEVICE)
print('BATCH      :', BATCH)
print('EPOCHS     :', EPOCHS)
print('IMGSZ      :', IMGSZ)
print('PATIENCE   :', PATIENCE)
print('MODELS     :', MODELS)


## 4) Merge dos datasets — normalmente NÃO rodar

A estrutura já existe em `/workspace/processed_4cls`, então esta célula fica comentada por padrão.

Rode apenas se você mudar os datasets de origem ou precisar reconstruir `merged_data`.


In [ ]:
# merge_script = DATA_DIR / 'merge_datasets.py'
#
# run_cmd([
#     sys.executable, str(merge_script),
#     '--taco_root', str(TACO_DIR),
#     '--external_root', str(EXTERNAL_DIR),
#     '--output_root', str(PROCESSED_DIR),
#     '--skip_preprocess',
# ])


## 5) Pré-processamento Path A — normalmente NÃO rodar

A estrutura `train/val/test` já existe em `/workspace/processed_4cls`, então esta célula fica comentada por padrão.

Rode apenas se você apagou/recriou o dataset processado ou mudou o mapeamento de classes.


In [ ]:
# preprocess_script = DATA_DIR / 'preprocess.py'
#
# run_cmd([
#     sys.executable, str(preprocess_script),
#     '--taco_root', str(PROCESSED_DIR / 'merged_data'),
#     '--output_root', str(PROCESSED_DIR),
#     '--path', 'A',
# ])


## 6) Conferência do dataset Path A

Confirme que o YAML está apontando para o dataset correto e que usa as 4 classes esperadas.


In [ ]:
import yaml

print('DATASET_YAML_PATH_A =', DATASET_YAML_PATH_A)
print('Existe?', DATASET_YAML_PATH_A.exists())

if DATASET_YAML_PATH_A.exists():
    with open(DATASET_YAML_PATH_A, 'r') as f:
        data_yaml = yaml.safe_load(f)
    print(data_yaml)


## 7) Limpeza opcional de cache `.npy` criado por `cache='disk'`

Se o volume cresceu muito depois de treinar, provavelmente foram criados arquivos `.npy` dentro de `processed_4cls`. Esta célula apenas mostra o tamanho; a deleção fica comentada.

Para evitar recriar isso, no `train_path_A.py` use `cache=False` ou `cache=True`, mas não `cache='disk'`.


In [ ]:
run_cmd('du -sh /workspace/processed_5cls || true')
run_cmd("find /workspace/processed_5cls -type f -name '*.npy' | wc -l")


# run_cmd("find /workspace/processed_5cls -type f -name '*.npy' -delete")
# run_cmd('du -sh /workspace/processed_5cls || true')

## 8) Treino Path A

Esta célula roda `train_path_A.py` com saída em `/workspace/runs/path_A`.

Importante: os pesos de **`yolov11m`**, **`yolov8m`** e **`yolov9s`** já foram calculados e estão em `runs`. Rode novamente apenas se quiser retreinar.


In [ ]:
train_script = TRAIN_DIR / 'train_path_A.py'

run_cmd([
    sys.executable, str(train_script),
    '--data', str(DATASET_YAML_PATH_A),
    '--output', str(RUNS_PATH_A_DIR),
    '--models', *MODELS,
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--device', str(DEVICE),
    '--patience', str(PATIENCE),
    '--mlflow_uri', str(MLFLOW_DIR),
])


## 9) Conferir pesos disponíveis

O avaliador espera encontrar os modelos no formato:

```text
/root/runs/path_A/<modelo>/weights/best.pt
```


In [ ]:
best_weights = sorted(RUNS_PATH_A_DIR.glob('*/weights/best.pt'))
print(f'Pesos encontrados em {RUNS_PATH_A_DIR}: {len(best_weights)}')
for p in best_weights:
    print(' -', p)

if not best_weights:
    raise FileNotFoundError(f'Nenhum best.pt encontrado em {RUNS_PATH_A_DIR}')


## 10) Resumo rápido dos treinos

In [ ]:
run_cmd([
    sys.executable, str(TRAIN_DIR / 'train_path_A.py'),
    '--summarize',
    '--output', str(RUNS_PATH_A_DIR),
])


## 11) Avaliação Path A

In [ ]:
evaluate_script = EVAL_DIR / 'evaluate.py'

run_cmd([
    sys.executable, str(evaluate_script),
    '--path', 'A',
    '--runs_dir', str(RUNS_PATH_A_DIR),
    '--data_yaml', str(DATASET_YAML_PATH_A),
    '--output', str(RESULTS_PATH_A_DIR),
    '--device', str(DEVICE),
    '--imgsz', str(IMGSZ),
])


## 12) Avaliação TTA + WBF — opcional

A célula abaixo fica comentada porque é mais lenta. Rode apenas se quiser avaliar com test-time augmentation e weighted boxes fusion.


In [ ]:
run_cmd([
    sys.executable, str(EVAL_DIR / 'evaluate.py'),
    '--path', 'A',
    '--runs_dir', str(RUNS_PATH_A_DIR),
    '--data_yaml', str(DATASET_YAML_PATH_A),
    '--output', str(RESULTS_PATH_A_DIR),
    '--device', str(DEVICE),
    '--imgsz', str(IMGSZ),
    "--use_tta_wbf",
    "--tta_scales", "512", "640", "768",
    "--tta_flip",
    "--tta_wbf_iou", "0.55",
    "--tta_skip_box_thr", "0.001",
])


## 13) Grid Search TTA + WBF

In [ ]:
import itertools
import subprocess
import sys

# Defina os ranges (ajuste conforme necessário)
tta_wbf_iou_values = [0.45, 0.50, 0.55, 0.60, 0.65]
tta_skip_box_thr_values = [0.0001, 0.001, 0.01, 0.05]

results = []

for iou, skip_thr in itertools.product(tta_wbf_iou_values, tta_skip_box_thr_values):
    print(f"Rodando com IOU={iou}, SKIP_THR={skip_thr}")

    cmd = [
        sys.executable, str(EVAL_DIR / 'evaluate_tta.py'),
        '--path', 'A',
        '--runs_dir', str(RUNS_PATH_A_DIR),
        '--data_yaml', str(DATASET_YAML_PATH_A),
        '--output', str(RESULTS_PATH_A_DIR / f"iou_{iou}_skip_{skip_thr}"),
        '--device', str(DEVICE),
        '--imgsz', str(IMGSZ),
        "--use_tta_wbf",
        "--tta_scales", "512", "640", "768",
        "--tta_flip",
        "--tta_wbf_iou", str(iou),
        "--tta_skip_box_thr", str(skip_thr),
    ]

    subprocess.run(cmd)

    # opcional: guardar configs testadas
    results.append({
        "iou": iou,
        "skip_thr": skip_thr,
        "output_dir": str(RESULTS_PATH_A_DIR / f"iou_{iou}_skip_{skip_thr}")
    })

print("Grid search finalizado!")

## 14) Checklist final

Antes de parar/deletar o pod, confirme que o que você quer manter está no volume:

- pesos essenciais em `/workspace/runs/path_A`
- resultados em `/workspace/results_path_A`
- projeto em `/workspace/TrashScan`
- dataset em `/workspace/processed_4cls`


In [ ]:
run_cmd('du -sh /workspace/runs /workspace/results_path_A /workspace/processed_4cls /workspace/TrashScan 2>/dev/null || true')
